# Load Data

In [68]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd

m = gp.Model("MIP")

from ranDataGen.order_data_loader import load_order_data

df = load_order_data(base_dir='ranDataGen/')

# df[df['order'] == 1]           
#df[df['tote'] == 1]            
# df.groupby('order').sum()      
# df.groupby('tote')['quantity'].sum()
df

,order,item_slot,itemtype,quantity,tote
0,1,0,3,3,1
1,1,1,1,2,1
2,2,0,2,3,2
3,2,1,3,1,3
4,2,2,0,1,2
5,3,0,3,3,3
6,3,1,2,3,2
7,3,2,0,1,1
8,4,0,1,1,0
9,4,1,2,1,0


# Position Based (Scroll Down for Actual Model)

In [62]:
# ==================================================
# Build inventory items: (item_id, item_type, tote)
# ==================================================

items = []
tote_items = {}

item_id = 0

for _, row in df.iterrows():
    i = row["itemtype"]
    t = row["tote"]
    qty = int(row["quantity"])
    
    for q in range(qty):
        item_id += 1
        item = (item_id, i, t)
        items.append(item)
        
        if t not in tote_items:
            tote_items[t] = []
        tote_items[t].append(item)

N = len(items)
positions = range(1, N+1)

totes = list(tote_items.keys())

# number of items per tote
tote_size = {t: len(tote_items[t]) for t in totes}

# ==================================================
# Build demand dictionary
# ==================================================

orders = df["order"].unique()

demand = {}

for _, row in df.iterrows():
    o = row["order"]
    i = row["itemtype"]
    qty = int(row["quantity"])
    
    demand[(o,i)] = demand.get((o,i),0) + qty

# ==================================================
# Item position on conveyor
# ==================================================

x = m.addVars(
    [(item_id,p) for (item_id,i,t) in items for p in positions],
    vtype=GRB.BINARY,
    name="x"
)

# Each item gets exactly one position

m.addConstrs(
    gp.quicksum(x[item_id,p] for p in positions) == 1
    for (item_id,i,t) in items
)

# Each position gets exactly one item

m.addConstrs(
    gp.quicksum(x[item_id,p] for (item_id,i,t) in items) == 1
    for p in positions
)

# ==================================================
# Tote contiguity constraints
# ==================================================

start = m.addVars(totes, vtype=GRB.INTEGER, lb=1, ub=N, name="tote_start")

# position expression for each item
pos_expr = {
    item_id: gp.quicksum(p * x[item_id,p] for p in positions)
    for (item_id,i,t) in items
}

# items must lie within tote block
for (item_id,i,t) in items:

    m.addConstr(pos_expr[item_id] >= start[t])

    m.addConstr(pos_expr[item_id] <= start[t] + tote_size[t] - 1)

# prevent tote overlap

z = m.addVars(totes, totes, vtype=GRB.BINARY, name="tote_order")

M = N

for t1 in totes:
    for t2 in totes:
        if t1 != t2:

            m.addConstr(
                start[t1] + tote_size[t1]
                <= start[t2] + M*(1 - z[t1,t2])
            )

            m.addConstr(
                start[t2] + tote_size[t2]
                <= start[t1] + M*(z[t1,t2])
            )

# ==================================================
# Belt assignment
# ==================================================

belts = range(1,5)

y = m.addVars(
    [(b,o) for b in belts for o in orders],
    vtype=GRB.BINARY,
    name="assign"
)

# each belt gets one order
m.addConstrs(
    gp.quicksum(y[b,o] for o in orders) == 1
    for b in belts
)

# each order goes to one belt
m.addConstrs(
    gp.quicksum(y[b,o] for b in belts) == 1
    for o in orders
)

# ==================================================
# Picking variables
# ==================================================

pick = m.addVars(
    [(b,item_id,o,p) for b in belts
                      for (item_id,i,t) in items
                      for o in orders
                      for p in positions],
    vtype=GRB.BINARY,
    name="pick"
)

m.addConstrs(
    pick[b,item_id,o,p] <= x[item_id,p]
    for b in belts
    for (item_id,i,t) in items
    for o in orders
    for p in positions
)

m.addConstrs(
    gp.quicksum(pick[b,item_id,o,p] for b in belts for o in orders) <= 1
    for (item_id,i,t) in items
    for p in positions
)

# belt precedence

m.addConstrs(
    pick[b+1,item_id,o,p] <=
    1 - gp.quicksum(pick[b,item_id,o2,p] for o2 in orders)
    
    for b in range(1,4)
    for (item_id,i,t) in items
    for o in orders
    for p in positions
)

# belt must match assigned order

m.addConstrs(
    pick[b,item_id,o,p] <= y[b,o]
    for b in belts
    for (item_id,i,t) in items
    for o in orders
    for p in positions
)

# remove invalid picks

for (item_id,i,t) in items:
    for o in orders:
        if (o,i) not in demand:
            for b in belts:
                for p in positions:
                    m.addConstr(pick[b,item_id,o,p] == 0)

# demand satisfaction

for (o,i),qty in demand.items():

    m.addConstr(
        gp.quicksum(
            pick[b,item_id,o,p]
            for b in belts
            for (item_id,it,t) in items if it == i
            for p in positions
        ) == qty
    )

# ==================================================
# Objective: minimize completion position
# ==================================================

T = m.addVar(vtype=GRB.INTEGER, name="makespan")

m.addConstrs(
    T >= p * pick[b,item_id,o,p]
    for b in belts
    for (item_id,i,t) in items
    for o in orders
    for p in positions
)

m.setObjective(T, GRB.MINIMIZE)

# ==================================================
# Solve
# ==================================================

m.optimize()

# ==================================================
# Print Results
# ==================================================

print("\nITEM SEQUENCE ON CONVEYOR")

sequence = []

for (item_id,i,t) in items:
    for p in positions:
        if x[item_id,p].X > 0.5:
            sequence.append((p,item_id,i,t))

sequence.sort()

for p,item_id,i,t in sequence:
    print(f"Position {p}: Item {item_id} | Type {i} | Tote {t}")

print("\nTOTE SEQUENCE")

for p,item_id,i,t in sequence:
    print(f"Position {p}: Tote {t}")

print("\nBELT → ORDER ASSIGNMENTS")

for b in belts:
    for o in orders:
        if y[b,o].X > 0.5:
            print(f"Belt {b} processes Order {o}")

print("\nPICKS")

for b in belts:
    for (item_id,i,t) in items:
        for o in orders:
            for p in positions:
                if pick[b,item_id,o,p].X > 0.5:
                    print(
                        f"Belt {b} picks Item {item_id} "
                        f"(Type {i}, Tote {t}) "
                        f"for Order {o} at position {p}"
                    )

print("\nSYSTEM COMPLETION POSITION:", T.X)

Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (mac64[x86] - Darwin 24.6.0 24G517)

CPU model: Intel(R) Core(TM) i7-1060NG7 CPU @ 1.20GHz
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 23963 rows, 6174 columns and 69454 nonzeros
Model fingerprint: 0xdc919fb0
Variable types: 0 continuous, 6174 integer (6169 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+01]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 2e+01]
  RHS range        [1e+00, 2e+01]
Presolve removed 19989 rows and 1850 columns
Presolve time: 2.44s
Presolved: 3974 rows, 4324 columns, 20830 nonzeros
Variable types: 0 continuous, 4324 integer (4319 binary)
Found heuristic solution: objective 19.0000000

Root relaxation: infeasible, 2482 iterations, 0.76 seconds (0.27 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0     0 infeas

# Time Based Model

In [71]:
# ==================================================
# Build inventory items: (item_id, item_type, tote)
# ==================================================

items = []
tote_items = {}

item_id = 0

for _, row in df.iterrows():
    i = row["itemtype"]
    t = row["tote"]
    qty = int(row["quantity"])
    
    for q in range(qty):
        item_id += 1
        item = (item_id, i, t)
        items.append(item)
        
        if t not in tote_items:
            tote_items[t] = []
        tote_items[t].append(item)

N = len(items)
positions = range(1, N+1)

totes = list(tote_items.keys())

# number of items per tote
tote_size = {t: len(tote_items[t]) for t in totes}

# ==================================================
# Build demand dictionary
# ==================================================

orders = df["order"].unique()

demand = {}

for _, row in df.iterrows():
    o = row["order"]
    i = row["itemtype"]
    qty = int(row["quantity"])
    
    demand[(o,i)] = demand.get((o,i),0) + qty

# ==================================================
# Item position on conveyor
# ==================================================

x = m.addVars(
    [(item_id,p) for (item_id,i,t) in items for p in positions],
    vtype=GRB.BINARY,
    name="x"
)

m.addConstrs(
    gp.quicksum(x[item_id,p] for p in positions) == 1
    for (item_id,i,t) in items
)

m.addConstrs(
    gp.quicksum(x[item_id,p] for (item_id,i,t) in items) == 1
    for p in positions
)

# ==================================================
# Tote contiguity constraints
# ==================================================

start = m.addVars(totes, vtype=GRB.INTEGER, lb=1, ub=N, name="tote_start")

pos_expr = {
    item_id: gp.quicksum(p * x[item_id,p] for p in positions)
    for (item_id,i,t) in items
}

for (item_id,i,t) in items:

    m.addConstr(pos_expr[item_id] >= start[t])
    m.addConstr(pos_expr[item_id] <= start[t] + tote_size[t] - 1)

# prevent tote overlap

z = m.addVars(totes, totes, vtype=GRB.BINARY, name="tote_order")

M = N

for t1 in totes:
    for t2 in totes:
        if t1 != t2:

            m.addConstr(
                start[t1] + tote_size[t1]
                <= start[t2] + M*(1 - z[t1,t2])
            )

            m.addConstr(
                start[t2] + tote_size[t2]
                <= start[t1] + M*(z[t1,t2])
            )

# ==================================================
# Belt assignment
# ==================================================

belts = range(1,5)

y = m.addVars(
    [(b,o) for b in belts for o in orders],
    vtype=GRB.BINARY,
    name="assign"
)

m.addConstrs(
    gp.quicksum(y[b,o] for o in orders) == 1
    for b in belts
)

m.addConstrs(
    gp.quicksum(y[b,o] for b in belts) == 1
    for o in orders
)

# ==================================================
# Picking variables
# ==================================================

pick = m.addVars(
    [(b,item_id,o,p) for b in belts
                      for (item_id,i,t) in items
                      for o in orders
                      for p in positions],
    vtype=GRB.BINARY,
    name="pick"
)

m.addConstrs(
    pick[b,item_id,o,p] <= x[item_id,p]
    for b in belts
    for (item_id,i,t) in items
    for o in orders
    for p in positions
)

m.addConstrs(
    gp.quicksum(pick[b,item_id,o,p] for b in belts for o in orders) <= 1
    for (item_id,i,t) in items
    for p in positions
)

# belt precedence

m.addConstrs(
    pick[b+1,item_id,o,p] <=
    1 - gp.quicksum(pick[b,item_id,o2,p] for o2 in orders)
    
    for b in range(1,4)
    for (item_id,i,t) in items
    for o in orders
    for p in positions
)

# belt must match assigned order

m.addConstrs(
    pick[b,item_id,o,p] <= y[b,o]
    for b in belts
    for (item_id,i,t) in items
    for o in orders
    for p in positions
)

# remove invalid picks

for (item_id,i,t) in items:
    for o in orders:
        if (o,i) not in demand:
            for b in belts:
                for p in positions:
                    m.addConstr(pick[b,item_id,o,p] == 0)

# demand satisfaction

for (o,i),qty in demand.items():

    m.addConstr(
        gp.quicksum(
            pick[b,item_id,o,p]
            for b in belts
            for (item_id,it,t) in items if it == i
            for p in positions
        ) == qty
    )

# ==================================================
# TIME-BASED OBJECTIVE
# ==================================================

TIME = m.addVar(vtype=GRB.CONTINUOUS, name="completion_time")

m.addConstrs(
    TIME >= (3*(p-1) + 7.5*(b-1) + 3) *
            pick[b,item_id,o,p]

    for b in belts
    for (item_id,i,t) in items
    for o in orders
    for p in positions
)

m.setObjective(TIME, GRB.MINIMIZE)

# ==================================================
# Solve
# ==================================================

m.optimize()

# ==================================================
# Print Results
# ==================================================

print("\nITEM SEQUENCE ON CONVEYOR")

sequence = []

for (item_id,i,t) in items:
    for p in positions:
        if x[item_id,p].X > 0.5:
            sequence.append((p,item_id,i,t))

sequence.sort()

for p,item_id,i,t in sequence:
    print(f"Position {p}: Item {item_id} | Type {i} | Tote {t}")

print("\nTOTE SEQUENCE")

for p,item_id,i,t in sequence:
    print(f"Position {p}: Tote {t}")

print("\nBELT → ORDER ASSIGNMENTS")

for b in belts:
    for o in orders:
        if y[b,o].X > 0.5:
            print(f"Belt {b} processes Order {o}")

print("\nPICKS WITH TIME")

for b in belts:
    for (item_id,i,t) in items:
        for o in orders:
            for p in positions:
                if pick[b,item_id,o,p].X > 0.5:

                    t_pick = 3*(p-1) + 7.5*(b-1) + 3

                    print(
                        f"Belt {b} picks Item {item_id} "
                        f"(Type {i}, Tote {t}) "
                        f"for Order {o} at position {p} "
                        f"→ Time = {t_pick:.2f} sec"
                    )

print("\nSYSTEM COMPLETION TIME:", TIME.X)

Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (mac64[x86] - Darwin 24.6.0 24G517)

CPU model: Intel(R) Core(TM) i7-1060NG7 CPU @ 1.20GHz
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 47926 rows, 12348 columns and 138908 nonzeros
Model fingerprint: 0x1d68219e
Variable types: 2 continuous, 12346 integer (12338 binary)
Coefficient statistics:
  Matrix range     [1e+00, 8e+01]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 2e+01]
  RHS range        [1e+00, 2e+01]

MIP start from previous solve produced solution with objective 64.5 (2.59s)
MIP start from previous solve produced solution with objective 61.5 (2.64s)
Processing MIP start from previous solve: 0 nodes explored in subMIP, total elapsed time 5s
Processing MIP start from previous solve: 0 nodes explored in subMIP, total elapsed time 10s
MIP start from previous solve produced solution with objective 57 (10.36s)
Loaded MIP start from previous solve with objective 5

# CSV Input File

In [72]:
# Shape mapping
shape_map = {
    0: "circle",
    1: "pentagon",
    2: "trapezoid",
    3: "triangle",
    4: "star",
    5: "moon",
    6: "heart",
    7: "cross"
}

belts = range(1,5)

rows = []

for b in belts:
    row = {"conv_num": b}

    # Initialize columns
    for col in shape_map.values():
        row[col] = 0

    # Count picks by item type
    for item_id, item_type, tote in items:

        if item_type not in shape_map:
            continue

        shape_name = shape_map[item_type]

        count = 0

        for o in orders:
            for p in positions:
                if pick[b, item_id, o, p].X > 0.5:
                    count += 1

        row[shape_name] += count

    rows.append(row)

# Create dataframe
df_picklist = pd.DataFrame(rows)

# Ensure correct column order
df_picklist = df_picklist[
    ["conv_num"] + list(shape_map.values())
]

# Save CSV
output_path = "MIP-belt_picklist.csv"
df_picklist.to_csv(output_path, index=False)

print("CSV generated:", output_path)

CSV generated: MIP-belt_picklist.csv
